In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from flax import nnx
import optax
import grain.python as grain

DATASET_PATH="./ranking_dataset.csv"
RANDOM_SEED=14

In [ ]:

class RoPE(nnx.Module):
    def __init__(self, head_dim: int, max_seq_len: int = 4096, base: float = 10000.0):
        if head_dim % 2 != 0:
            raise ValueError(f"head_dim must be even. Got {head_dim}")
            
        # Compute angles
        inv_freq = 1.0 / (base ** (jnp.arange(0, head_dim, 2) / head_dim))
        positions = jnp.arange(max_seq_len)
        angles = jnp.outer(positions, inv_freq)
        
        # Cache complex exponentials e^(i * theta)
        complex_freqs = jnp.exp(1j * angles)
        self.complex_freqs = nnx.Cache(complex_freqs)

    def __call__(self, x: jax.Array) -> jax.Array:
        """
        Expects x to have shape: (batch_size, seq_len, n_heads, head_dim)
        """
        seq_len = x.shape[1]
        freqs = self.complex_freqs.value[:seq_len, :]
        
        # 2. Reshape for broadcasting: (1, seq_len, 1, head_dim // 2)
        freqs = freqs[None, :, None, :]
        
        # 3. Group head_dim into pairs: (..., head_dim // 2, 2)
        x_paired = x.reshape(*x.shape[:-1], -1, 2)
        
        # 4. Map to complex plane and multiply
        x_complex = jax.lax.complex(x_paired[..., 0], x_paired[..., 1])
        rotated_complex = x_complex * freqs
        
        # 5. Map back to real coordinates and flatten
        rotated_real = jnp.stack([rotated_complex.real, rotated_complex.imag], axis=-1)
        return rotated_real.reshape(x.shape)

In [ ]:
class FeedForwardNN(nnx.Module):
    def __init__(self, d_model: int, hidden_dim: int,rngs: nnx.Rngs):
        self.w_gate= nnx.Linear(d_model, hidden_dim, use_bias=False, rngs=rngs)
        self.w_up= nnx.Linear(d_model, hidden_dim, use_bias=False, rngs=rngs)
        self.w_down= nnx.Linear(hidden_dim,d_model, use_bias=False, rngs=rngs)
    def __call__(self, x:jax.Array):
        hidden_state=self.w_up(x)*jax.nn.silu(self.w_gate(x))
        return self.w_down(hidden_state)

In [ ]:
class MHSA(nnx.Module):
    def __init__(self, d_model,max_seq_len, d_k=None, n_heads=8,rngs=None):
        
        self.d_model = d_model
        self.n_heads=n_heads
        if d_k is not None: self.d_k=d_k
        else: self.d_k=d_model
        self.head_size=self.d_k//n_heads
        self.max_seq=max_seq_len
        # Linear layers for projecting input X into Q, K, V spaces
        self.w_q = nnx.Linear(d_model, self.d_k, use_bias=False, rngs=rngs)
        self.w_k = nnx.Linear(d_model, self.d_k, use_bias=False, rngs=rngs)
        self.w_v = nnx.Linear(d_model, self.d_k, use_bias=False, rngs=rngs)
        self.w_o = nnx.Linear(self.d_k, d_model, use_bias=False, rngs=rngs)
        self.RoPE=RoPE(head_dim=self.head_size,max_seq_len=max_seq_len)
        
    def __call__(self, x: jax.Array, mask: jax.Array | None = None) -> jax.Array:
        """
        x shape: (batch_size, seq_len, d_model)
        """
        batch,seq_len ,_ = x.shape
        # 1. Project inputs
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)
        Q=q.reshape((batch,seq_len,self.n_heads,self.head_size))
        K=k.reshape((batch,seq_len,self.n_heads,self.head_size))
        V=v.reshape((batch,seq_len,self.n_heads,self.head_size))
        #apply RoPE
        Q_R=self.RoPE(Q)
        K_R=self.RoPE(K)

        #Permute matrix for computing Attention tensor
        Q_R = jnp.transpose(Q_R, (0, 2, 1, 3))
        K_R = jnp.transpose(K_R, (0, 2, 1, 3))
        V = jnp.transpose(V, (0, 2, 1, 3))
        
        # Scaled dot-product scores: (batch, n_heads, seq_len, seq_len)
        scale = 1.0 / jnp.sqrt(self.head_size)
        attn_scores = jnp.matmul(Q_R, jnp.transpose(K_R, (0, 1, 3, 2))) * scale
        if mask is not None:
            attn_scores = jnp.where(mask, attn_scores, -1e12)
        attn_weights = jax.nn.softmax(attn_scores, axis=-1)
        self.sown_attn = nnx.Intermediate(attn_weights)
        # Weighted sum over values: (batch, n_heads, seq_len, head_size)
        context = jnp.matmul(attn_weights, V)
        
        # Permute back to (batch, seq_len, n_heads, head_size) and flatten heads
        context = jnp.transpose(context, (0, 2, 1, 3))
        attn_output = context.reshape((batch, seq_len, self.d_k))
        
        # Final projection
        return self.w_o(attn_output)

In [ ]:


class TransformerBlock(nnx.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, rngs: nnx.Rngs):
        self.attn_norm = nnx.RMSNorm(d_model, rngs=rngs)
        self.mha = MHSA(d_model, n_heads=n_heads, max_seq_len=max_seq_len, rngs=rngs)
        
        self.ffn_norm = nnx.RMSNorm(d_model, rngs=rngs)
        hidden_dim = int((8/3) * d_model) 
        self.ffn = FeedForwardNN(d_model, hidden_dim, rngs)

    def __call__(self, x: jax.Array, mask: jax.Array | None = None) -> jax.Array:
        """
        x shape: (batch_size, seq_len, d_model)
        """
        # Phase 1: Pre-Norm Attention Residual Stream
        x_norm1 = self.attn_norm(x)
        attn_out = self.mha(x_norm1, mask)
        x = x + attn_out  # First residual addition
        
        # Phase 2: Pre-Norm FeedForward Residual Stream
        x_norm2 = self.ffn_norm(x)
        ffn_out = self.ffn(x_norm2)
        x = x + ffn_out  # Second residual addition
        
        return x

In [ ]:
# Step 1: DigitRanker Transformer Encoder
import jax
import jax.numpy as jnp
import numpy as np
from flax import nnx
import optax
from typing import Any

class DigitRanker(nnx.Module):
    def __init__(self, d_model: int = 64, n_heads: int = 4, num_layers: int = 2, max_seq_len: int = 10, num_classes: int = 10, rngs: nnx.Rngs = None):
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.num_classes = num_classes
        self.embed = nnx.Linear(1, d_model, use_bias=True, rngs=rngs)
        self.pos_embed = nnx.Embed(max_seq_len, d_model, rngs=rngs)
        self.transformer_blocks = [
            TransformerBlock(d_model, n_heads, max_seq_len, rngs) for _ in range(num_layers)
        ]
        self.norm = nnx.RMSNorm(d_model, rngs=rngs)
        self.head = nnx.Linear(d_model, num_classes, use_bias=True, rngs=rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        # x: (batch, seq_len) -- integer values
        # Min-max normalize per sequence
        x_min = x.min(axis=-1, keepdims=True)
        x_max = x.max(axis=-1, keepdims=True)
        x_norm = (x - x_min) / (x_max - x_min + 1e-8)
        x_norm = x_norm[..., None]  # (batch, seq_len, 1)
        x_emb = self.embed(x_norm)  # (batch, seq_len, d_model)
        pos_ids = jnp.arange(self.max_seq_len)[None, :]
        pos_emb = self.pos_embed(pos_ids)
        x = x_emb + pos_emb
        for block in self.transformer_blocks:
            x = block(x)
        x = self.norm(x)
        logits = self.head(x)  # (batch, seq_len, num_classes)
        return logits


In [ ]:
# Step 2: Load dataset, split, and move to GPU
import pandas as pd
from sklearn.model_selection import train_test_split
import jax
import jax.numpy as jnp

# Load CSV
df = pd.read_csv("./ranking_dataset.csv")
X = np.stack(df['input'].apply(lambda s: np.fromstring(s, sep=' ')).values)
y = np.stack(df['target'].apply(lambda s: np.fromstring(s, sep=' ')).values)

# Split into train, val, test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Move to GPU (JAX arrays)
X_train = jax.device_put(X_train)
y_train = jax.device_put(y_train)
X_val = jax.device_put(X_val)
y_val = jax.device_put(y_val)
X_test = jax.device_put(X_test)
y_test = jax.device_put(y_test)


In [ ]:
# Step 3: get_batch function
def get_batch(X, y, batch_size, key=None):
    n = X.shape[0]
    if key is None:
        idx = np.random.choice(n, batch_size, replace=False)
    else:
        idx = jax.random.choice(key, n, (batch_size,), replace=False)
    return X[idx], y[idx]


In [ ]:
# Step 4: Loss, train_step, validation_step
import optax

def compute_loss(logits, targets):
    # logits: (batch, seq_len, num_classes), targets: (batch, seq_len)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, targets.astype(jnp.int32))
    return loss.mean()

def train_step(model, params, opt_state, batch, optimizer):
    X, y = batch
    def loss_fn(params):
        logits = model.apply(params, X)
        loss = compute_loss(logits, y)
        return loss, logits
    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(params)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss, logits

def validation_step(model, params, batch):
    X, y = batch
    logits = model.apply(params, X)
    loss = compute_loss(logits, y)
    return loss, logits


In [ ]:
# Step 5: Model, optimizer, scheduler, EarlyStopping
import time

# Model init
d_model = 64
n_heads = 4
num_layers = 2
max_seq_len = 10
num_classes = 10
rng = jax.random.PRNGKey(42)

model = DigitRanker(d_model, n_heads, num_layers, max_seq_len, num_classes, rngs=nnx.Rngs(rng))
params = model.init(rng, X_train[:2])  # dummy batch for shape

# Cosine decay scheduler
lr = 1e-3
scheduler = optax.cosine_decay_schedule(init_value=lr, decay_steps=1000, alpha=0.1)
optimizer = optax.adamw(scheduler)
opt_state = optimizer.init(params)

# EarlyStopping
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.best_params = None
    def step(self, val_loss, params):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_params = params
        else:
            self.counter += 1
        return self.counter >= self.patience


In [ ]:
# Step 6: Training loop
batch_size = 64
epochs = 100
patience = 10
steps_per_epoch = X_train.shape[0] // batch_size

es = EarlyStopping(patience=patience)

for epoch in range(epochs):
    t0 = time.time()
    train_losses = []
    for step in range(steps_per_epoch):
        batch = get_batch(X_train, y_train, batch_size)
        params, opt_state, loss, _ = train_step(model, params, opt_state, batch, optimizer)
        train_losses.append(loss)
    val_batch = (X_val, y_val)
    val_loss, _ = validation_step(model, params, val_batch)
    print(f"Epoch {epoch+1} | Train Loss: {np.mean(train_losses):.4f} | Val Loss: {val_loss:.4f} | Time: {time.time()-t0:.2f}s")
    if es.step(val_loss, params):
        print("Early stopping triggered.")
        params = es.best_params
        break


In [ ]:
# Step 7: Evaluation on test set
from sklearn.metrics import accuracy_score

def evaluate(model, params, X, y):
    logits = model.apply(params, X)
    preds = jnp.argmax(logits, axis=-1)
    acc = (preds == y).mean()
    # Exact sequence accuracy
    exact_seq_acc = ((preds == y).all(axis=1)).mean()
    return float(acc), float(exact_seq_acc)

test_acc, test_exact_acc = evaluate(model, params, X_test, y_test)
print(f"Test Token Accuracy: {test_acc:.4f}")
print(f"Test Exact Sequence Accuracy: {test_exact_acc:.4f}")
